# Skin Lesion **Segmentation** — PAD-UFES-20 (YOLOv8n-seg)
**Week 4 deliverable:** instance segmentation code + IoU/Dice evaluation.

Project 1 (Patient Skin Lesion Monitoring), Computer Vision Semester 6.

PAD-UFES-20 has no ground-truth masks, so we generate **pseudo-masks** with
classical image processing (LAB color + Otsu + largest central blob),
convert them to YOLO polygon labels, and train **YOLOv8n-seg** (COCO
pretrained) for instance segmentation across the 6 diagnostic classes
(BCC, SCC, ACK, SEK, MEL, NEV).

## How to use
1. Runtime → Change runtime type → **GPU (T4)** → Save.
2. Make sure `images.zip` and `metadata.csv` are in your Drive folder
   `skin-lesion-monitoring-cv_dataset`.
3. Run top to bottom. Best weights are saved to
   `pad-ufes-20-results/best_yolov8_seg.pt`.
4. Download that file next to `segment_app.py` to run the local app.

In [ ]:
# 1. Imports + GPU check
import os, glob, zipfile, time, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4).')

In [ ]:
# 2. Install ultralytics (YOLOv8)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'ultralytics==8.2.103'], check=True)
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 4. Config
DATASET_DIR   = '/content/drive/MyDrive/skin-lesion-monitoring-cv_dataset'
ZIP_ON_DRIVE  = os.path.join(DATASET_DIR, 'images.zip')
META_ON_DRIVE = os.path.join(DATASET_DIR, 'metadata.csv')
DATA_ROOT     = '/content/pad-ufes-20'
YOLO_ROOT     = '/content/yolo_seg_data'
OUT_DIR       = '/content/drive/MyDrive/pad-ufes-20-results'

IMG_SIZE   = 640
BATCH_SIZE = 16
EPOCHS     = 25
VAL_SPLIT  = 0.2
SEED       = 42
CLASSES    = ['BCC', 'SCC', 'ACK', 'SEK', 'MEL', 'NEV']
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(ZIP_ON_DRIVE),  f'Zip not found: {ZIP_ON_DRIVE}'
assert os.path.isfile(META_ON_DRIVE), f'metadata.csv not found: {META_ON_DRIVE}'

In [ ]:
# 5. Extract dataset to local disk
local_zip  = '/content/images.zip'
local_meta = '/content/metadata.csv'

t = time.time()
shutil.copy(ZIP_ON_DRIVE, local_zip)
shutil.copy(META_ON_DRIVE, local_meta)
print(f'Copied in {time.time() - t:.0f}s')

t = time.time()
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(DATA_ROOT)
shutil.copy(local_meta, os.path.join(DATA_ROOT, 'metadata.csv'))
print(f'Extracted in {time.time() - t:.0f}s')

paths = {}
for ext in ('*.png', '*.jpg', '*.jpeg'):
    for p in glob.glob(os.path.join(DATA_ROOT, '**', ext), recursive=True):
        paths[os.path.basename(p)] = p
meta = pd.read_csv(os.path.join(DATA_ROOT, 'metadata.csv'))
meta = meta[['img_id', 'diagnostic']].dropna()
meta = meta[meta['diagnostic'].isin(CLASSES)].copy()
meta['path'] = meta['img_id'].map(paths)
meta = meta.dropna(subset=['path']).reset_index(drop=True)
print('Usable samples:', len(meta))

In [ ]:
# 6. Auto-generate pseudo-masks + convert to YOLO polygon labels
def pseudo_mask(img_bgr, min_area_frac=0.005):
    h, w = img_bgr.shape[:2]
    lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = cv2.GaussianBlur(lab[..., 0], (7, 7), 0)
    Linv = 255 - L
    _, th = cv2.threshold(Linv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN,  k, iterations=1)

    n, lab_im, stats, cents = cv2.connectedComponentsWithStats(th, 8)
    if n <= 1:
        return None
    img_cx, img_cy = w / 2, h / 2
    best_i, best_score = -1, -1.0
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        af = area / (w * h)
        if af < min_area_frac or af > 0.92:
            continue
        cx, cy = cents[i]
        d = np.hypot(cx - img_cx, cy - img_cy) / np.hypot(img_cx, img_cy)
        score = af * (1.0 - 0.6 * d)
        if score > best_score:
            best_score, best_i = score, i
    if best_i < 0:
        return None
    mask = (lab_im == best_i).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
    return mask

def mask_to_yolo_polygon(mask, w, h, max_points=80):
    """Largest contour -> normalized polygon points list (flat: x1 y1 x2 y2 ...)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        return None
    # Reduce points with Douglas-Peucker so labels stay compact
    eps = 0.002 * cv2.arcLength(cnt, True)
    cnt = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2)
    if len(cnt) < 3:
        return None
    if len(cnt) > max_points:
        idx = np.linspace(0, len(cnt) - 1, max_points).astype(int)
        cnt = cnt[idx]
    poly = []
    for x, y in cnt:
        poly.append(max(0.0, min(1.0, x / w)))
        poly.append(max(0.0, min(1.0, y / h)))
    return poly

rows = []
kept = skipped = 0
t = time.time()
for r in meta.itertuples():
    im = cv2.imread(r.path)
    if im is None:
        skipped += 1; continue
    h, w = im.shape[:2]
    m = pseudo_mask(im)
    if m is None:
        skipped += 1; continue
    poly = mask_to_yolo_polygon(m, w, h)
    if poly is None:
        skipped += 1; continue
    rows.append({'img': r.path, 'cls': r.diagnostic, 'poly': poly,
                 'mask': m, 'h': h, 'w': w, 'img_id': r.img_id})
    kept += 1
print(f'Generated {kept} polygons (skipped {skipped}) in {time.time() - t:.0f}s')

# Visualize a few
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
shown = random.sample(rows, 8)
for ax, row in zip(axes.ravel(), shown):
    im = cv2.cvtColor(cv2.imread(row['img']), cv2.COLOR_BGR2RGB)
    over = im.copy(); over[row['mask'] > 0] = (255, 64, 96)
    blend = cv2.addWeighted(im, 0.6, over, 0.4, 0)
    ax.imshow(blend); ax.set_title(row['cls']); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 7. Build YOLO segmentation directory structure
from sklearn.model_selection import train_test_split

df_full = pd.DataFrame([{'img_id': r['img_id'], 'img': r['img'],
                         'cls': r['cls'], 'poly': r['poly']}
                        for r in rows])
train_df, val_df = train_test_split(df_full, test_size=VAL_SPLIT,
                                    stratify=df_full['cls'], random_state=SEED)
print('train:', len(train_df), ' val:', len(val_df))

# Clean + rebuild
if os.path.isdir(YOLO_ROOT):
    shutil.rmtree(YOLO_ROOT)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    os.makedirs(os.path.join(YOLO_ROOT, sub), exist_ok=True)

def write_split(df_, split):
    img_dir = os.path.join(YOLO_ROOT, 'images', split)
    lab_dir = os.path.join(YOLO_ROOT, 'labels', split)
    for r in df_.itertuples():
        # symlink to save disk
        dst = os.path.join(img_dir, r.img_id)
        if not os.path.exists(dst):
            try:
                os.symlink(r.img, dst)
            except OSError:
                shutil.copy(r.img, dst)
        stem = os.path.splitext(r.img_id)[0]
        cls_idx = CLS_TO_IDX[r.cls]
        line = ' '.join([str(cls_idx)] + [f'{p:.6f}' for p in r.poly])
        with open(os.path.join(lab_dir, stem + '.txt'), 'w') as f:
            f.write(line + '\n')

write_split(train_df, 'train')
write_split(val_df,   'val')

data_yaml = os.path.join(YOLO_ROOT, 'data.yaml')
with open(data_yaml, 'w') as f:
    f.write(f"path: {YOLO_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {len(CLASSES)}\n")
    f.write('names: [' + ', '.join(f"'{c}'" for c in CLASSES) + ']\n')
print('Wrote', data_yaml)

In [ ]:
# 8. Train YOLOv8n-seg (COCO pretrained)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'wandb'])

import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
os.chdir('/content')

from ultralytics import YOLO
model = YOLO('yolov8n-seg.pt')
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    seed=SEED,
    project='runs_seg',
    name='lesion_seg',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    save_period=5,          # save a checkpoint every 5 epochs
    verbose=True,
)
print('Training complete.')

In [ ]:
# 9. Validate + compute IoU/Dice on val pseudo-masks, save best to Drive
best_src = '/content/runs_seg/lesion_seg/weights/best.pt'
best_dst = os.path.join(OUT_DIR, 'best_yolov8_seg.pt')
shutil.copy(best_src, best_dst)
print('Saved:', best_dst)

best = YOLO(best_src)
val_metrics = best.val(data=data_yaml, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                       split='val', verbose=False)
box_map50    = float(val_metrics.box.map50)
box_map50_95 = float(val_metrics.box.map)
seg_map50    = float(val_metrics.seg.map50)
seg_map50_95 = float(val_metrics.seg.map)
print(f'box mAP@50    : {box_map50:.4f}')
print(f'box mAP@50-95 : {box_map50_95:.4f}')
print(f'seg mAP@50    : {seg_map50:.4f}')
print(f'seg mAP@50-95 : {seg_map50_95:.4f}')

# Pixel-level IoU + Dice vs pseudo-masks (binary, lesion-vs-bg)
def compute_pixel_metrics(model, df_, eps=1e-6):
    ious, dices = [], []
    for r in df_.itertuples():
        im = cv2.imread(r.img)
        if im is None: continue
        h, w = im.shape[:2]
        # rebuild GT pseudo-mask
        gt = pseudo_mask(im)
        if gt is None: continue
        gt_bin = (gt > 127).astype(np.uint8)
        # predict
        res = model.predict(im, imgsz=IMG_SIZE, conf=0.25, iou=0.45, verbose=False)[0]
        pred = np.zeros((h, w), dtype=np.uint8)
        if res.masks is not None and len(res.masks.data):
            for mk in res.masks.data.cpu().numpy():
                mk_r = cv2.resize((mk > 0.5).astype(np.uint8),
                                  (w, h), interpolation=cv2.INTER_NEAREST)
                pred |= mk_r
        inter = int(((pred == 1) & (gt_bin == 1)).sum())
        union = int(((pred == 1) | (gt_bin == 1)).sum())
        ps    = int((pred == 1).sum())
        gs    = int((gt_bin == 1).sum())
        ious.append((inter + eps) / (union + eps))
        dices.append((2 * inter + eps) / (ps + gs + eps))
    return float(np.mean(ious)), float(np.mean(dices))

val_iou, val_dice = compute_pixel_metrics(best, val_df)
print(f'Pixel IoU  : {val_iou:.4f}')
print(f'Pixel Dice : {val_dice:.4f}')

with open(os.path.join(OUT_DIR, 'metrics_yolov8_seg.txt'), 'w') as f:
    f.write('YOLOv8n-seg  PAD-UFES-20 segmentation (pseudo-mask supervision)\n')
    f.write(f'box mAP@50    : {box_map50:.4f}\n')
    f.write(f'box mAP@50-95 : {box_map50_95:.4f}\n')
    f.write(f'seg mAP@50    : {seg_map50:.4f}\n')
    f.write(f'seg mAP@50-95 : {seg_map50_95:.4f}\n')
    f.write(f'Pixel IoU     : {val_iou:.4f}\n')
    f.write(f'Pixel Dice    : {val_dice:.4f}\n')

In [ ]:
# 10. Visualize predictions  (image | pseudo-mask | YOLOv8-seg overlay)
best = YOLO(best_dst)

sample = val_df.sample(6, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(6, 3, figsize=(11, 18))
for i, row in sample.iterrows():
    im_bgr = cv2.imread(row['img'])
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    h, w = im_bgr.shape[:2]
    gt = pseudo_mask(im_bgr)
    res = best.predict(im_bgr, imgsz=IMG_SIZE, conf=0.25, iou=0.45, verbose=False)[0]
    over = im_rgb.copy()
    if res.masks is not None and len(res.masks.data):
        for mk, cls in zip(res.masks.data.cpu().numpy(),
                            res.boxes.cls.cpu().numpy().astype(int)):
            mk_r = cv2.resize((mk > 0.5).astype(np.uint8),
                              (w, h), interpolation=cv2.INTER_NEAREST)
            color = np.array([255, 64, 96], dtype=np.uint8)
            over[mk_r > 0] = color
    blend = cv2.addWeighted(im_rgb, 0.55, over, 0.45, 0)
    axes[i, 0].imshow(im_rgb); axes[i, 0].set_title('image');       axes[i, 0].axis('off')
    axes[i, 1].imshow(gt, cmap='gray'); axes[i, 1].set_title('pseudo-mask'); axes[i, 1].axis('off')
    axes[i, 2].imshow(blend); axes[i, 2].set_title('YOLOv8-seg');    axes[i, 2].axis('off')
plt.tight_layout(); plt.show()
print('Done. Download best_yolov8_seg.pt from Drive next to segment_app.py.')